# Introduction to Pandas
## A Hands-On Tutorial for Movement Neuroscience Graduate Students

---

**Why this tutorial?**  
Pandas is how you handle **tabular data** in Python — think spreadsheets, CSV files, trial logs, and participant metadata. If NumPy is for raw numerical arrays, Pandas is for labeled, structured data where rows and columns have names and meaning. Every time you load a CSV of trial results, compute summary statistics by condition, merge data from multiple sessions, or clean up missing values from dropped sensors, you'll use Pandas.

This notebook covers the two core Pandas data structures — **Series** and **DataFrame** — along with file I/O, indexing, filtering, missing data handling, merging, reshaping, arithmetic, apply/map, sorting, descriptive statistics, hierarchical indexing, and groupby aggregation.

**Prerequisites:** Python Basics (L0), NumPy (L1), Data Structures & Functions (L2).  
**Environment:** Google Colab (recommended) or Jupyter Notebook.

---

## Table of Contents

**Part I — Series**
1. [Setup and Reading/Writing Files](#1)
2. [Creating a Series](#2)
3. [Indexing, Slicing, and Filtering Series](#3)
4. [`loc` vs `iloc` — Label vs Integer Indexing](#4)

**Part II — DataFrames**
5. [Creating a DataFrame](#5)
6. [Accessing Columns and Rows](#6)
7. [Indexing with `loc` and `iloc`](#7)
8. [Boolean Indexing and Conditional Assignment](#8)
9. [Reindexing and Converting to NumPy](#9)
10. [Useful Utilities — `value_counts`, `unique`, `rename`, `corr`](#10)

**Part III — Data Cleaning**
11. [Handling Missing Data — `isna`, `dropna`, `fillna`, `interpolate`](#11)
12. [Sorting — by Index and by Values](#12)

**Part IV — Combining and Reshaping**
13. [Merging and Joining DataFrames](#13)
14. [Concatenating DataFrames](#14)
15. [Reshaping — `pivot_table`, `melt`, `stack`/`unstack`](#15)

**Part V — Operations and Analysis**
16. [Arithmetic Operations and Broadcasting](#16)
17. [Applying Functions with `apply` and `applymap`](#17)
18. [Descriptive Statistics](#18)
19. [Hierarchical Indexing (MultiIndex)](#19)
20. [GroupBy — Split-Apply-Combine](#20)

**Part VI — Practice**
21. [Exercises](#21)
22. [Summary & Further Reading](#22)

In [ ]:
# ---- Setup: import libraries ----

import pandas as pd
import numpy as np

# These are used so frequently that importing them directly is common:
from pandas import Series, DataFrame

---
# Part I — Series

## 1. Setup and Reading/Writing Files <a id='1'></a>

Before diving into data structures, let's cover **file I/O** — the first thing you do in any analysis script.

### Reading Files

| Function | File type | Common parameters |
|---|---|---|
| `pd.read_csv()` | CSV, TSV, text | `sep`, `header`, `index_col`, `usecols`, `skiprows`, `na_values` |
| `pd.read_excel()` | Excel (.xlsx, .xls) | `sheet_name`, `header`, `usecols` |
| `pd.read_table()` | Tab-separated | Same as `read_csv` with `sep='\t'` |

In [ ]:
# ---- Load a CSV file with Pandas (Colab-specific) ----
# Adapt the path for your workspace.

import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks KINES 597/data/')
!pwd

data = pd.read_csv("GazeVar50.csv")     # Load CSV into a DataFrame
data.describe()                          # Quick statistical summary

In [ ]:
# ---- Useful read_csv parameters ----
# (These examples use generic paths — adapt for your files)

# Read only specific columns:
# df = pd.read_csv("data.csv", usecols=["participant", "rt", "condition"])

# Use a specific column as the index:
# df = pd.read_csv("data.csv", index_col="trial_id")

# Skip the first 3 rows (e.g., metadata lines from a sensor export):
# df = pd.read_csv("data.csv", skiprows=3)

# Read a tab-separated file:
# df = pd.read_csv("data.tsv", sep="\t")

# Treat specific values as NaN (common in lab software exports):
# df = pd.read_csv("data.csv", na_values=["N/A", "-999", "missing"])

### Writing Files

In [ ]:
# ---- Saving DataFrames to files ----

# Save to CSV (most common):
# df.to_csv("results.csv", index=False)   # index=False avoids writing the index as a column

# Save to Excel:
# df.to_excel("results.xlsx", sheet_name="Sheet1", index=False)

# Example: create a small DataFrame and save it
demo = pd.DataFrame({"participant": ["P01", "P02"], "mean_rt": [0.31, 0.38]})
# demo.to_csv("demo_output.csv", index=False)
print("Demo DataFrame:")
print(demo)

Pandas has two workhorse data structures:

| Structure | Dimensions | Analogy |
|---|---|---|
| **Series** | 1-D | A single column of a spreadsheet (with row labels) |
| **DataFrame** | 2-D | An entire spreadsheet (rows × columns, both labeled) |

---
## 2. Creating a Series <a id='2'></a>

A **Series** is a 1-D array-like object containing a sequence of values and an associated array of labels called the **index**.

In [ ]:
# ---- Create a Series from a list ----
# Pandas auto-generates an integer index (0, 1, 2, ...) if you don't provide one.

s = pd.Series([1, 3, 5, np.nan, 6, 8])
print(s)
print(f"\nValues: {s.array}")    # The underlying data
print(f"Index:  {s.index}")      # The index labels

In [ ]:
# ---- Create a Series with custom string labels ----

obj = pd.Series([4, 7, -5, 3], index=["d", "b", "a", "c"])
print(obj)
print(f"\nIndex: {obj.index}")

# Access and modify elements by label:
obj["a"] = 9
print(f"\nAfter obj['a'] = 9:\n{obj}")

# Check membership:
print(f"\n'e' in obj: {'e' in obj}")

In [ ]:
# ---- IMPORTANT: assignment creates a REFERENCE, not a copy ----

obj2 = obj          # obj2 and obj point to the SAME data
obj2["a"] = 6
print(f"obj['a']  = {obj['a']}")   # Also changed!
print(f"obj2['a'] = {obj2['a']}")

In [ ]:
# ---- Reindexing: rearrange or add new index labels ----

obj = pd.Series([4.5, 7.2, -5.3, 3.6], index=["d", "b", "a", "c"])
obj2 = obj.reindex(["a", "b", "c", "d", "e"])  # 'e' doesn't exist → NaN
print("Reindexed:")
print(obj2)

In [ ]:
# ---- Convert between dictionaries and Series ----

sdata = {"Ohio": 35000, "Texas": 71000, "Oregon": 16000, "Utah": 5000}
obj3 = pd.Series(sdata)       # Dict → Series
print(f"Series:\n{obj3}")

d1_dict = obj3.to_dict()      # Series → Dict
print(f"\nBack to dict: {d1_dict}")

---
## 3. Indexing, Slicing, and Filtering Series <a id='3'></a>

In [ ]:
# ---- Indexing by label and by integer position ----

obj = pd.Series(np.arange(4.), index=["a", "b", "c", "d"])
print(obj)
print(f"\nobj['b'] = {obj['b']}")
print(f"obj['b'] == obj[1]: {obj['b'] == obj[1]}")

In [ ]:
# ---- Slicing, fancy indexing, and Boolean filtering ----

print(f"obj[2:4]:\n{obj[2:4]}")
print(f"\nobj[[1, 3]]:\n{obj[[1, 3]]}")
print(f"\nobj[obj < 2]:\n{obj[obj < 2]}")

---
## 4. `loc` vs `iloc` — Label vs Integer Indexing <a id='4'></a>

| Accessor | Indexes by | Endpoint inclusive? |
|---|---|---|
| **`loc`** | **Labels** (strings, dates, etc.) | Yes |
| **`iloc`** | **Integer positions** (0, 1, 2, ...) | No (like Python slicing) |

**Always use `loc` or `iloc` explicitly** to avoid ambiguity.

In [ ]:
# ---- loc: select by label (endpoint INCLUSIVE) ----

obj2 = pd.Series([1, 2, 3], index=["a", "b", "c"])
print(f"obj2.loc['b':'c']:\n{obj2.loc['b':'c']}")  # Includes 'c'!

obj2.loc["b":"c"] = 5
print(f"\nAfter assignment:\n{obj2}")

In [ ]:
# ---- iloc ALWAYS uses integer positions ----

obj1 = pd.Series([1, 2, 3], index=[2, 0, 1])   # Integer index!
obj2 = pd.Series([1, 2, 3], index=["a", "b", "c"])

print(f"obj1.iloc[[0, 1, 2]]: {list(obj1.iloc[[0, 1, 2]])}")
print(f"obj2.iloc[[0, 1, 2]]: {list(obj2.iloc[[0, 1, 2]])}")

---
# Part II — DataFrames

## 5. Creating a DataFrame <a id='5'></a>

In [ ]:
# ---- Create a DataFrame from a dictionary ----

data_dict = {"state": ["Ohio", "Ohio", "Ohio", "Nevada", "Nevada", "Nevada"],
             "year": [2000, 2001, 2002, 2001, 2002, 2003],
             "pop": [1.5, 1.7, 3.6, 2.4, 2.9, 3.2]}

frame = pd.DataFrame(data_dict)
print(frame)
print(f"\nhead():\n{frame.head()}")   # First 5 rows
print(f"\ntail():\n{frame.tail(3)}")  # Last 3 rows

In [ ]:
# ---- Create from nested dicts; transpose ----

populations = {"Ohio": {2000: 1.5, 2001: 1.7, 2002: 3.6},
               "Nevada": {2001: 2.4, 2002: 2.9}}

frame3 = pd.DataFrame(populations)
print(f"DataFrame:\n{frame3}")
print(f"\nTransposed:\n{frame3.T}")

---
## 6. Accessing Columns and Rows <a id='6'></a>

In [ ]:
# ---- Access columns, create/delete columns ----

frame2 = pd.DataFrame(data_dict, columns=["year", "state", "pop"])

# Two ways to access a column:
x = frame2["state"]       # Bracket notation (always works)
y = frame2.state           # Dot notation (only for valid Python names)

# Create new columns:
frame2["debt"] = np.arange(6.)                   # By array assignment
frame2["eastern"] = frame2["state"] == "Ohio"     # Boolean column
print(frame2)

# Delete a column:
del frame2["eastern"]
print(f"\nColumns: {list(frame2.columns)}")

**Important:** The column returned from indexing a DataFrame is a **view**, not a copy. Modifications will be reflected in the DataFrame. Use `.copy()` if you need an independent copy.

---
## 7. Indexing with `loc` and `iloc` on DataFrames <a id='7'></a>

On DataFrames, you can specify **both rows and columns** separated by a comma: `df.loc[rows, columns]`.

In [ ]:
# ---- Sample DataFrame for indexing ----

data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                     index=["Ohio", "Colorado", "Utah", "New York"],
                     columns=["one", "two", "three", "four"])
print(data)

In [ ]:
# ---- loc: rows and columns by LABEL ----

print(data.loc[["Colorado", "New York"]])
print(f"\nColorado, two & three:\n{data.loc['Colorado', ['two', 'three']]}")

In [ ]:
# ---- iloc: rows and columns by INTEGER POSITION ----

print(data.iloc[[1, 2], [3, 0, 1]])
print(f"\nLast row, first 2 cols:\n{data.iloc[-1, 0:2]}")

---
## 8. Boolean Indexing and Conditional Assignment <a id='8'></a>

Boolean indexing works with `loc` (but **not** with `iloc`).

In [ ]:
# ---- Boolean mask and assignment ----

data[data < 5] = 0
print("After data[data < 5] = 0:")
print(data)

# Targeted: set only column 'three' where four > 5
data.loc[data["four"] > 5] = 3
print(f"\nAfter conditional row assignment:\n{data}")

data.loc[data.three == 3, "three"] = 6   # Modify just one column
print(f"\nAfter targeted column assignment:\n{data}")

---
## 9. Reindexing and Converting to NumPy <a id='9'></a>

In [ ]:
# ---- Reindexing rows and columns ----

frame = pd.DataFrame(np.arange(9).reshape((3, 3)),
                      index=["a", "c", "d"],
                      columns=["Ohio", "Texas", "California"])

frame2 = frame.reindex(index=["a", "b", "c", "d"])  # 'b' → NaN row
print(f"Reindexed rows:\n{frame2}")

frame3 = frame.reindex(columns=["Texas", "Utah", "California"])
print(f"\nReindexed columns (Ohio dropped, Utah = NaN):\n{frame3}")

# Convert to NumPy:
arr = frame3.to_numpy()
print(f"\nNumPy array:\n{arr}")

---
## 10. Useful Utilities — `value_counts`, `unique`, `rename`, `corr` <a id='10'></a>

These are the everyday tools you'll reach for constantly when exploring experimental data.

In [ ]:
# ---- Create sample trial data for demonstration ----

rng = np.random.default_rng(42)
trials = pd.DataFrame({
    "participant": np.repeat(["P01", "P02", "P03"], 20),
    "condition": np.tile(["control", "control", "perturb", "perturb", "washout"] * 4, 3),
    "rt": rng.normal(0.35, 0.05, 60),
    "error_cm": rng.exponential(1.5, 60)
})
print(f"Shape: {trials.shape}")
print(trials.head(10))

In [ ]:
# ---- value_counts(): count occurrences of each unique value ----
# Essential for checking your experimental design (are all conditions balanced?).

print("Trials per condition:")
print(trials["condition"].value_counts())

print("\nTrials per participant:")
print(trials["participant"].value_counts())

In [ ]:
# ---- unique() and nunique(): distinct values ----

print(f"Unique conditions: {trials['condition'].unique()}")
print(f"Number of unique participants: {trials['participant'].nunique()}")

In [ ]:
# ---- rename(): rename columns or index labels ----
# Common when column names from sensor software are cryptic.

trials_renamed = trials.rename(columns={"rt": "reaction_time_s",
                                         "error_cm": "endpoint_error_cm"})
print(trials_renamed.columns.tolist())

In [ ]:
# ---- corr(): compute pairwise correlations between numeric columns ----
# Quick way to check if RT and error are related.

print("Correlation matrix:")
print(trials[["rt", "error_cm"]].corr().round(3))

---
# Part III — Data Cleaning

## 11. Handling Missing Data — `isna`, `dropna`, `fillna`, `interpolate` <a id='11'></a>

Missing data is inevitable in experiments: sensors drop out, trials are aborted, participants skip conditions. Pandas uses `NaN` (Not a Number) to represent missing values.

| Function | What it does |
|---|---|
| `df.isna()` / `df.notna()` | Boolean mask of missing/present values |
| `df.isna().sum()` | Count missing values per column |
| `df.dropna()` | Remove rows (or columns) with missing values |
| `df.fillna(value)` | Replace NaN with a specific value |
| `df.interpolate()` | Fill NaN by interpolating between neighbors |

In [ ]:
# ---- Create data with missing values ----

messy = pd.DataFrame({
    "trial": [1, 2, 3, 4, 5, 6, 7, 8],
    "rt": [0.31, 0.29, np.nan, 0.34, 0.28, np.nan, 0.33, 0.30],
    "force_N": [45.2, np.nan, 48.1, 50.3, np.nan, np.nan, 44.7, 46.8],
    "condition": ["ctrl", "ctrl", "ctrl", "ctrl", "pert", "pert", "pert", "pert"]
})
print(messy)

In [ ]:
# ---- Detect missing data ----

print("Missing values (True = NaN):")
print(messy.isna())

print(f"\nCount per column:\n{messy.isna().sum()}")
print(f"\nTotal missing: {messy.isna().sum().sum()}")

In [ ]:
# ---- dropna(): remove rows with ANY missing value ----
# Use how='all' to only drop rows where ALL values are NaN.

clean = messy.dropna()             # Default: drop rows with any NaN
print(f"After dropna(): {len(clean)} rows (was {len(messy)})")
print(clean)

# Drop only rows where 'rt' specifically is NaN:
clean_rt = messy.dropna(subset=["rt"])
print(f"\nAfter dropna(subset=['rt']): {len(clean_rt)} rows")

In [ ]:
# ---- fillna(): replace NaN with a specific value ----

# Fill with a constant:
filled_zero = messy["force_N"].fillna(0)
print(f"Filled with 0:\n{filled_zero.values}")

# Fill with the column mean (a common imputation strategy):
filled_mean = messy["force_N"].fillna(messy["force_N"].mean())
print(f"\nFilled with mean ({messy['force_N'].mean():.1f}):\n{filled_mean.values}")

# Forward fill (carry the last valid value forward):
filled_ffill = messy["force_N"].ffill()
print(f"\nForward filled:\n{filled_ffill.values}")

In [ ]:
# ---- interpolate(): fill NaN by linear interpolation ----
# Ideal for time-series data where values change smoothly.

interpolated = messy["force_N"].interpolate()
print(f"Interpolated:\n{interpolated.values}")

**Best practices for missing data:**
- **Always check first** with `isna().sum()` before any analysis.
- Use `dropna()` when missing values are few and random.
- Use `fillna(mean)` or `interpolate()` for sensor dropouts in time-series.
- Document and report how you handled missing data in your methods section.

---
## 12. Sorting — by Index and by Values <a id='12'></a>

In [ ]:
# ---- sort_index(): sort by row or column labels ----

frame = pd.DataFrame(np.arange(8).reshape((2, 4)),
                      index=["three", "one"],
                      columns=["d", "a", "b", "c"])
print(f"Original:\n{frame}")
print(f"\nSort rows:\n{frame.sort_index()}")
print(f"\nSort columns:\n{frame.sort_index(axis='columns')}")

In [ ]:
# ---- sort_values(): sort by one or more column values ----
# This is used FAR more often than sort_index in practice.

# Sort trials by reaction time (ascending):
print("Sorted by RT (fastest first):")
print(trials.sort_values("rt").head(5))

# Sort by multiple columns:
print("\nSorted by condition, then RT:")
print(trials.sort_values(["condition", "rt"]).head(10))

# Descending order:
print("\nSlowest RTs:")
print(trials.sort_values("rt", ascending=False).head(5))

---
# Part IV — Combining and Reshaping

## 13. Merging and Joining DataFrames <a id='13'></a>

`pd.merge()` is the Pandas equivalent of a SQL JOIN. It combines two DataFrames based on **shared columns** (keys).

**Neuroscience use:** Merge trial-level data with participant demographics, or combine kinematic data with EMG timestamps.

| Merge type | What it keeps |
|---|---|
| `how='inner'` | Only rows with matching keys in **both** DataFrames (default) |
| `how='left'` | All rows from the left DataFrame; NaN where right has no match |
| `how='right'` | All rows from the right DataFrame |
| `how='outer'` | All rows from **both**; NaN where no match |

In [ ]:
# ---- Merging two DataFrames on a shared column ----

# Participant demographics:
demographics = pd.DataFrame({
    "participant": ["P01", "P02", "P03", "P04"],
    "age": [24, 31, 27, 22],
    "handedness": ["right", "left", "right", "right"]
})

# Trial-level summary (some participants may not have data):
trial_summary = pd.DataFrame({
    "participant": ["P01", "P02", "P03"],
    "mean_rt": [0.312, 0.398, 0.287],
    "n_trials": [48, 50, 45]
})

print("Demographics:")
print(demographics)
print("\nTrial summary:")
print(trial_summary)

In [ ]:
# ---- Inner merge (default): only participants in BOTH tables ----

merged_inner = pd.merge(demographics, trial_summary, on="participant")
print(f"Inner merge ({len(merged_inner)} rows):")
print(merged_inner)

In [ ]:
# ---- Left merge: keep ALL demographics, fill NaN where no trial data ----

merged_left = pd.merge(demographics, trial_summary, on="participant", how="left")
print(f"Left merge ({len(merged_left)} rows):")
print(merged_left)  # P04 has NaN for mean_rt and n_trials

In [ ]:
# ---- Merge on different column names ----
# If the key columns have different names in each DataFrame:

# pd.merge(df1, df2, left_on="subject_id", right_on="participant")

---
## 14. Concatenating DataFrames <a id='14'></a>

`pd.concat()` stacks DataFrames vertically (adding rows) or horizontally (adding columns). Common when combining data from multiple participants or sessions.

| Parameter | Effect |
|---|---|
| `axis=0` | Stack vertically (add rows) — default |
| `axis=1` | Stack horizontally (add columns) |
| `ignore_index=True` | Reset the index after concatenation |

In [ ]:
# ---- Concatenate: stack participant data vertically ----

p01 = pd.DataFrame({"trial": [1, 2, 3], "rt": [0.31, 0.29, 0.34], "pid": "P01"})
p02 = pd.DataFrame({"trial": [1, 2, 3], "rt": [0.38, 0.42, 0.39], "pid": "P02"})

combined = pd.concat([p01, p02], ignore_index=True)  # Reset index to 0..5
print(combined)

In [ ]:
# ---- Concatenate horizontally: add columns side by side ----

kinematics = pd.DataFrame({"peak_vel": [1.2, 1.5, 1.1]}, index=[0, 1, 2])
emg = pd.DataFrame({"biceps_rms": [0.45, 0.52, 0.41]}, index=[0, 1, 2])

combined_h = pd.concat([kinematics, emg], axis=1)
print(combined_h)

---
## 15. Reshaping — `pivot_table`, `melt`, `stack`/`unstack` <a id='15'></a>

Experimental data often needs to be reshaped between **wide format** (one column per condition) and **long format** (one row per observation).

| Direction | Function | Use case |
|---|---|---|
| Long → Wide | `pivot_table()` | Summary table for a report |
| Wide → Long | `melt()` | Prepare data for Seaborn or statsmodels |
| Columns → Rows | `stack()` | Compress column levels |
| Rows → Columns | `unstack()` | Expand row levels |

In [ ]:
# ---- pivot_table: long → wide summary ----
# Compute mean RT per participant × condition.

pivot = trials.pivot_table(
    values="rt",                # What to aggregate
    index="participant",        # Rows
    columns="condition",        # Columns
    aggfunc="mean"              # Aggregation function
)
print("Pivot table (mean RT):")
print(pivot.round(3))

In [ ]:
# ---- melt: wide → long ----
# Convert the pivot table back to long format (needed for Seaborn/stats).

wide_data = pd.DataFrame({
    "participant": ["P01", "P02", "P03"],
    "control_rt": [0.31, 0.38, 0.29],
    "perturb_rt": [0.42, 0.45, 0.40],
    "washout_rt": [0.35, 0.39, 0.32]
})
print(f"Wide format:\n{wide_data}")

long_data = wide_data.melt(
    id_vars="participant",          # Column(s) to keep as identifiers
    var_name="condition",            # Name for the new "variable" column
    value_name="rt"                  # Name for the new "value" column
)
print(f"\nLong format:\n{long_data}")

In [ ]:
# ---- stack / unstack: for MultiIndex DataFrames ----

print("Pivot table:")
print(pivot.round(3))

stacked = pivot.stack()   # Wide → long (columns become a row index level)
print(f"\nStacked:\n{stacked.round(3)}")

unstacked = stacked.unstack()  # Long → wide (back to original)
print(f"\nUnstacked:\n{unstacked.round(3)}")

---
# Part V — Operations and Analysis

## 16. Arithmetic Operations and Broadcasting <a id='16'></a>

Pandas arithmetic aligns data by **index labels** before operating.

In [ ]:
# ---- Element-wise arithmetic and broadcasting ----

frame = pd.DataFrame(np.arange(12.).reshape((4, 3)),
                      columns=list("bde"),
                      index=["Utah", "Ohio", "Texas", "Oregon"])
series = frame.iloc[0]   # First row as a Series

print(f"frame:\n{frame}")
print(f"\nframe - series (subtract first row from all):\n{frame - series}")

In [ ]:
# ---- NumPy ufuncs work directly with Pandas ----

frame = pd.DataFrame(np.random.standard_normal((4, 3)),
                      columns=list("bde"),
                      index=["Utah", "Ohio", "Texas", "Oregon"])
print(f"np.abs(frame):\n{np.abs(frame).round(3)}")
print(f"\nColumn maxima: {frame.max().round(3).to_dict()}")

---
## 17. Applying Functions with `apply` and `applymap` <a id='17'></a>

In [ ]:
# ---- apply: run a function on each column or row ----

def range_func(x):
    return x.max() - x.min()

def min_max(x):
    return pd.Series([x.min(), x.max()], index=["min", "max"])

print(f"Range per column:\n{frame.apply(range_func).round(3)}")
print(f"\nMin/Max per row:\n{frame.apply(min_max, axis='columns').round(3)}")

In [ ]:
# ---- applymap: function on every element ----

print(frame.applymap(lambda x: f"{x:.2f}"))

---
## 18. Descriptive Statistics <a id='18'></a>

In [ ]:
# ---- Descriptive stats with NaN handling ----

df = pd.DataFrame([[1.4, np.nan], [7.1, -4.5], [np.nan, np.nan], [0.75, -1.3]],
                   index=["a", "b", "c", "d"],
                   columns=["one", "two"])
print(f"df:\n{df}")
print(f"\nRow sums (skipna=True):\n{df.sum(axis='columns', skipna=True)}")
print(f"\nIndex of max per column:\n{df.idxmax()}")
print(f"\nFull describe:\n{df.describe()}")

---
## 19. Hierarchical Indexing (MultiIndex) <a id='19'></a>

In [ ]:
# ---- MultiIndex on rows AND columns ----

frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                      index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
                      columns=[["Ohio", "Ohio", "Colorado"],
                               ["Green", "Red", "Green"]])
frame.index.names = ["key1", "key2"]
frame.columns.names = ["state", "color"]
print(frame)

print(f"\nSwapped levels:\n{frame.swaplevel('key1', 'key2')}")
print(f"\nSum by color:\n{frame.groupby(level='color', axis='columns').sum()}")

---
## 20. GroupBy — Split-Apply-Combine <a id='20'></a>

**GroupBy** splits data into groups, applies a function, and combines results.

**Neuroscience use case:** Mean RT per condition, per participant, or per condition × block.

In [ ]:
# ---- GroupBy with the trials DataFrame ----

print(f"Mean RT by condition:\n{trials.groupby('condition')['rt'].mean().round(3)}")
print(f"\nMean RT by participant × condition:")
print(trials.groupby(["participant", "condition"])["rt"].mean().round(3))

In [ ]:
# ---- Multiple aggregations at once ----

summary = trials.groupby("condition")["rt"].agg(["mean", "std", "count"])
print(summary.round(3))

In [ ]:
# ---- Iterate over groups ----

for name, group in trials.groupby("condition"):
    print(f"Condition: '{name}' — {len(group)} trials, mean RT = {group['rt'].mean():.3f}")

---
# Part VI — Practice

## 21. Exercises <a id='21'></a>

---

### Exercise 1: Create and Filter a Participant DataFrame

Create a DataFrame with columns `ID`, `Age`, `Handedness`, `Mean_RT` for 4 participants. Then:
1. Set `ID` as the index.
2. Select only right-handed participants using Boolean indexing.
3. Compute the mean age and mean RT of the selected participants.

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 2: Handle Missing Data

Given:
```python
sensor = pd.DataFrame({
    "time_s": np.arange(0, 1, 0.1),
    "force_N": [12.1, 13.4, np.nan, np.nan, 15.8, 16.2, np.nan, 17.1, 18.3, 19.0]
})
```

1. Count the missing values.
2. Create a version with NaN rows dropped.
3. Create a version with NaN filled by linear interpolation.
4. Plot both the dropped and interpolated versions to see the difference.

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 3: Merge and Analyze

1. Create a `demographics` DataFrame (participant, age, group) and a `results` DataFrame (participant, mean_rt, mean_error).
2. Merge them on `participant` using a left join.
3. Use `groupby('group')` to compute the mean RT and error per group.

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 4: Reshape Wide ↔ Long

Given wide-format data:
```python
wide = pd.DataFrame({
    "participant": ["P01", "P02", "P03"],
    "pre_rt": [0.35, 0.39, 0.33],
    "post_rt": [0.28, 0.31, 0.27]
})
```

1. Use `melt()` to convert to long format with columns: `participant`, `timepoint`, `rt`.
2. Use `pivot_table()` to convert back to wide format.
3. Compute the mean RT per timepoint using `groupby`.

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 5: Full Pipeline (Challenge)

Simulate a realistic mini-analysis:
1. Create trial data for 3 participants × 3 conditions × 10 trials each (90 rows).
2. Inject 5 random `NaN` values into the RT column.
3. Use `isna().sum()` to report how many are missing.
4. Drop the NaN rows.
5. Use `groupby` + `agg` to compute mean, std, and count per participant × condition.
6. Use `pivot_table` to create a wide summary.
7. Save the result to a CSV.

In [ ]:
# ---- YOUR CODE HERE ----


---
## 22. Summary & Further Reading <a id='22'></a>

### What You Learned

| Concept | What it does | Example |
|---|---|---|
| **`read_csv` / `to_csv`** | Read and write CSV files | `pd.read_csv("data.csv", usecols=[...])` |
| **Series** | 1-D labeled array | `pd.Series([1,2,3], index=['a','b','c'])` |
| **DataFrame** | 2-D labeled table | `pd.DataFrame({'col': [...]})` |
| **`loc`** | Index by **labels** (inclusive) | `df.loc['Ohio', 'two':'four']` |
| **`iloc`** | Index by **integers** (exclusive) | `df.iloc[0:2, 1:3]` |
| **Boolean indexing** | Filter by condition | `df[df['rt'] > 0.3]` |
| **`value_counts`** | Count occurrences | `df['condition'].value_counts()` |
| **`unique` / `nunique`** | Distinct values | `df['pid'].nunique()` |
| **`rename`** | Rename columns/index | `df.rename(columns={'old': 'new'})` |
| **`corr`** | Pairwise correlations | `df[['rt','error']].corr()` |
| **`isna` / `dropna` / `fillna`** | Handle missing data | `df.dropna(subset=['rt'])` |
| **`interpolate`** | Fill NaN by interpolation | `df['force'].interpolate()` |
| **`sort_values`** | Sort by column values | `df.sort_values('rt')` |
| **`merge`** | Join two DataFrames by key | `pd.merge(df1, df2, on='pid')` |
| **`concat`** | Stack DataFrames | `pd.concat([p01, p02])` |
| **`pivot_table`** | Long → wide summary | `df.pivot_table(values='rt', ...)` |
| **`melt`** | Wide → long | `df.melt(id_vars='pid')` |
| **`apply` / `applymap`** | Custom functions | `df.apply(np.mean, axis=0)` |
| **`describe`** | Summary statistics | `df.describe()` |
| **MultiIndex** | Hierarchical labels | Multi-level rows/columns |
| **`groupby`** | Split-apply-combine | `df.groupby('cond')['rt'].mean()` |
| **`agg`** | Multiple aggregations | `.agg(['mean','std','count'])` |

### Further Reading

- [Pandas Getting Started](https://pandas.pydata.org/docs/getting_started/intro_tutorials/) — official tutorials
- [10 Minutes to Pandas](https://pandas.pydata.org/docs/user_guide/10min.html) — quick overview
- [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) — printable reference
- [Real Python — Pandas Merge](https://realpython.com/pandas-merge-join-and-concat/) — deep dive on combining data
- [Real Python — Pandas GroupBy](https://realpython.com/pandas-groupby/) — deep dive on aggregation
- *Python for Data Analysis* by Wes McKinney — the definitive Pandas book

### What's Next?

With Pandas mastered, you're ready for:
- **Seaborn** — statistical visualization built on Pandas DataFrames
- **SciPy** — signal processing and statistical tests
- **Statsmodels** — regression, ANOVA, mixed-effects models
- **scikit-learn** — machine learning and dimensionality reduction

---

*Happy wrangling, and may your DataFrames always be tidy!*